<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/13_Algoritmos_de_Busca_BFS_e_DFS_em_Tubulacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 13 - Notebook: Busca em Largura (BFS) e Busca em Profundidade (DFS) na Linha de Envase

Neste notebook implementamos os algoritmos **BFS** e **DFS** para descoberta de caminhos de fluido e enumeração de rotas alternativas de contingência na **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)**.


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=3.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino,
            "Comprimento (m)": comprimento_m, "Válvula ISA": tag_valvula, "Diâmetro (pol)": diametro_pol
        })

def criar_rede_padrao():
    nos = [
        "TS1_Suprimento", "TS2_Auxiliar", "VS1_Succao",
        "BC1_Bomba", "BC2_Bomba", "AS1_Acumulador",
        "VS2_Envase", "SQ2_Medicao", "EST_Envase"
    ]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("TS1_Suprimento", "VS1_Succao", 5.0, "VS1", 3.0)
    g.adicionar_tubulacao("TS2_Auxiliar", "VS1_Succao", 6.0, "VS1_AUX", 3.0)
    g.adicionar_tubulacao("VS1_Succao", "BC1_Bomba", 3.0, "SP1_SQ1", 3.0)
    g.adicionar_tubulacao("VS1_Succao", "BC2_Bomba", 4.0, "SP1_B_SQ1", 3.0)
    g.adicionar_tubulacao("BC1_Bomba", "AS1_Acumulador", 8.0, "CHECK_V1", 2.5)
    g.adicionar_tubulacao("BC2_Bomba", "AS1_Acumulador", 10.0, "CHECK_V2", 2.5)
    g.adicionar_tubulacao("AS1_Acumulador", "VS2_Envase", 10.0, "VS2", 2.0)
    g.adicionar_tubulacao("VS2_Envase", "SQ2_Medicao", 2.0, "SQ2_IN", 1.5)
    g.adicionar_tubulacao("AS1_Acumulador", "SQ2_Medicao", 14.0, "XV_BYPASS", 2.0)
    g.adicionar_tubulacao("SQ2_Medicao", "EST_Envase", 1.5, "BICO_FILL", 1.5)
    return g

from collections import deque
from typing import List, Set, Optional

rede = criar_rede_padrao()

class NavegadorGrafos:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def bfs_menor_numero_valvulas(self, origem: str, destino: str,
                                  nos_bloqueados: Optional[Set[str]] = None) -> Optional[List[str]]:
        if nos_bloqueados is None: nos_bloqueados = set()
        if origem in nos_bloqueados or destino in nos_bloqueados: return None

        fila = deque([(origem, [origem])])
        visitados = {origem}
        while fila:
            u_nome, caminho = fila.popleft()
            if u_nome == destino: return caminho
            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if self.g.adj_binaria[u_idx][v_idx] == 1 and v_nome not in visitados and v_nome not in nos_bloqueados:
                    visitados.add(v_nome)
                    fila.append((v_nome, caminho + [v_nome]))
        return None

    def dfs_todos_os_caminhos(self, origem: str, destino: str,
                             nos_bloqueados: Optional[Set[str]] = None) -> List[List[str]]:
        if nos_bloqueados is None: nos_bloqueados = set()
        caminhos = []
        def dfs(u_nome: str, visitados: List[str]):
            if u_nome == destino:
                caminhos.append(list(visitados))
                return
            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if self.g.adj_binaria[u_idx][v_idx] == 1 and v_nome not in visitados and v_nome not in nos_bloqueados:
                    visitados.append(v_nome)
                    dfs(v_nome, visitados)
                    visitados.pop()
        dfs(origem, [origem])
        return caminhos

nav = NavegadorGrafos(rede)
rota_bfs = nav.bfs_menor_numero_valvulas("TS1_Suprimento", "EST_Envase")
print("1. Rota com menor número de válvulas (BFS):", " -> ".join(rota_bfs))

todas_rotas = nav.dfs_todos_os_caminhos("TS1_Suprimento", "EST_Envase")
print(f"\n2. Total de opções de rotas (DFS): {len(todas_rotas)}")
for i, r in enumerate(todas_rotas, 1):
    print(f"  Opção {i}: {' -> '.join(r)}")

rota_cont = nav.bfs_menor_numero_valvulas("TS1_Suprimento", "EST_Envase", nos_bloqueados={"BC1_Bomba"})
print("\n3. Rota com contingência na Bomba BC1_Bomba:", " -> ".join(rota_cont))
assert "BC2_Bomba" in rota_cont


1. Rota com menor número de válvulas (BFS): TS1_Suprimento -> VS1_Succao -> BC1_Bomba -> AS1_Acumulador -> SQ2_Medicao -> EST_Envase

2. Total de opções de rotas (DFS): 4
  Opção 1: TS1_Suprimento -> VS1_Succao -> BC1_Bomba -> AS1_Acumulador -> VS2_Envase -> SQ2_Medicao -> EST_Envase
  Opção 2: TS1_Suprimento -> VS1_Succao -> BC1_Bomba -> AS1_Acumulador -> SQ2_Medicao -> EST_Envase
  Opção 3: TS1_Suprimento -> VS1_Succao -> BC2_Bomba -> AS1_Acumulador -> VS2_Envase -> SQ2_Medicao -> EST_Envase
  Opção 4: TS1_Suprimento -> VS1_Succao -> BC2_Bomba -> AS1_Acumulador -> SQ2_Medicao -> EST_Envase

3. Rota com contingência na Bomba BC1_Bomba: TS1_Suprimento -> VS1_Succao -> BC2_Bomba -> AS1_Acumulador -> SQ2_Medicao -> EST_Envase
